# F1 Strategist — Colab Smoke Notebook

Smoke-test version of the training pipeline. Runs in **~7 minutes on a Colab Free T4**.

**Owner:** Person 2 (Tanish). Fill cells in Phase 3.

**Goal:** demonstrate that the env + TRL stack work end-to-end on commodity hardware. The full 500-step run goes on the RTX 5090; this notebook is the W1 minimum requirement (Colab-runnable training script).

## What this notebook does

1. Install OpenEnv, TRL, and supporting deps (~3 min)
2. Pull the F1 Strategist env via `from_hf_space("Deltasthic/f1-strategist")` (Docker pulled into Colab)
3. Smoke train: 30 GRPO steps on Qwen3-0.6B (~3 min)
4. Plot the reward curve
5. Print one full rollout transcript

## What this notebook does NOT do

- The full 500-step run (needs ≥32 GB VRAM, won't fit on T4)
- Final eval on held-out seeds (use `evaluate.py` on the GPU server)
- Postmortem ablation (separate notebook or use the GPU server)

## 1. Install dependencies

Pin to versions known-good with Qwen3 + TRL 0.14. See `TRAINING.md` for the full rationale.

In [ ]:
# TODO Phase 3, Person 2:
# %pip install -q openenv-core>=0.2.3
# %pip install -q 'trl==0.14.0' 'transformers>=4.55.4' 'peft>=0.13.2' 'datasets>=3.2.0' 'accelerate>=1.3'
# %pip install -q 'liger-kernel<0.6' 'bitsandbytes>=0.49' matplotlib

## 2. Connect to the F1 Strategist env

Two options:
- **From the published HF Space:** zero local setup
- **From a local Docker build:** if you cloned the repo into Colab

In [ ]:
# TODO Phase 3, Person 2:
# from openenv import OpenEnvClient
# from client import F1StrategistEnv
# env = F1StrategistEnv.from_hf_space('Deltasthic/f1-strategist')
# obs = env.reset(options={'task': 'dry_strategy_sprint', 'seed': 0})
# print(obs)

## 3. Smoke-train Qwen3-0.6B with GRPO

30 steps, batch=1, grad-accum=4. Should take ~3 minutes on T4.

In [ ]:
# TODO Phase 3, Person 2:
# from trl import GRPOTrainer, GRPOConfig
# config = GRPOConfig(
#     model_name='Qwen/Qwen3-0.6B',
#     reward_funcs=['environment'],
#     max_steps=30,
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=4,
#     save_steps=10,
#     logging_steps=2,
#     output_dir='./grpo_colab_smoke',
# )
# def env_factory(): return F1StrategistEnv.from_hf_space('Deltasthic/f1-strategist')
# trainer = GRPOTrainer(config=config, env_factory=env_factory)
# trainer.train()

## 4. Plot the reward curve

In [ ]:
# TODO Phase 3, Person 2:
# import matplotlib.pyplot as plt
# import json
# state = json.load(open('./grpo_colab_smoke/trainer_state.json'))
# logs = state['log_history']
# steps = [l['step'] for l in logs if 'reward' in l]
# rewards = [l['reward'] for l in logs if 'reward' in l]
# plt.plot(steps, rewards)
# plt.xlabel('step'); plt.ylabel('reward'); plt.title('GRPO smoke (30 steps, T4)')
# plt.show()

## 5. Print one rollout transcript

Run the (just-trained) policy through one episode and print every (obs, action, reward) triple.

In [ ]:
# TODO Phase 3, Person 2:
# from inference import run_inference
# run_inference(model='./grpo_colab_smoke/checkpoint-30',
#               task='dry_strategy_sprint',
#               n_episodes=1,
#               verbose=True)

## What's next

If this notebook ran cleanly:
- The env is reachable from Colab
- The TRL stack imports without conflict
- The reward signal is non-zero (you should see the curve trend up over 30 steps)

For real numbers, run on a 32 GB+ GPU via [`TRAINING.md`](https://github.com/Deltasthicc/f1-strategist/blob/main/TRAINING.md):

```bash
python train.py --model Qwen/Qwen3-4B --max-steps 500 --batch-size 1 --grad-accum 32
```